## В данной задаче делаю предсказание возраста опоссума

### Загружаю датасет с https://www.kaggle.com/datasets/abrambeyer/openintro-possum

In [1]:
pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

dataset_name = "abrambeyer/openintro-possum"
save_path = os.path.expanduser("~/Py/DLS_mipt/PossumRegression/kaggle_data")

os.makedirs(save_path, exist_ok=True)
os.system(f"kaggle datasets download -d {dataset_name} -p {save_path} --unzip")


Dataset URL: https://www.kaggle.com/datasets/abrambeyer/openintro-possum
License(s): CC0-1.0


0

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
df = pd.read_csv("kaggle_data/possum.csv")
df.head()

,case,site,Pop,sex,age,hdlngth,skullw,totlngth,taill,footlgth,earconch,eye,chest,belly
0,1,1,Vic,m,8.0,94.1,60.4,89.0,36.0,74.5,54.5,15.2,28.0,36.0
1,2,1,Vic,f,6.0,92.5,57.6,91.5,36.5,72.5,51.2,16.0,28.5,33.0
2,3,1,Vic,f,6.0,94.0,60.0,95.5,39.0,75.4,51.9,15.5,30.0,34.0
3,4,1,Vic,f,6.0,93.2,57.1,92.0,38.0,76.1,52.2,15.2,28.0,34.0
4,5,1,Vic,f,2.0,91.5,56.3,85.5,36.0,71.0,53.2,15.1,28.5,33.0


### Подготовка датасета
#### site, Pop, case, sex - допускаю, что влияния нет

In [4]:
df.drop(columns=['site', 'Pop', 'case', 'sex'], inplace = True)

In [5]:
df.isna().sum()

age         2
hdlngth     0
skullw      0
totlngth    0
taill       0
footlgth    1
earconch    0
eye         0
chest       0
belly       0
dtype: int64

In [6]:
#удаляем Nan
df.dropna(inplace=True)

In [7]:
X = df.drop(columns=['age']).values
y = df['age'].values

In [8]:
!pip uninstall -y scikit-learn # удалим более старую версию библиотеки
!pip install scikit-learn

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Defaulting to user installation because normal site-packages is not writeable
  Using cached scikit_learn-1.6.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.5 MB)


In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, test_size=0.2, random_state=44)

### kNN алгоритм

In [10]:
from math import sqrt
def calc_distance(vec1, vec2):
    distance = 0.0
    for i in range(len(vec1)):
        distance += (vec1[i] - vec2[i])**2
    return sqrt(distance)

In [11]:
calc_distance(X_train[0], X_test[0])

15.320574401764446

In [12]:
def get_k_neighbour(train, test_row, num_neighbour):
    distances = []
    nearest_neighbour_ids = []
    for train_id, train_row in enumerate(train):
        distance_train_and_test = calc_distance(train_row, test_row)
        distances.append((train_id, distance_train_and_test))

    distances.sort(key=lambda x: x[1])
    for i in range(num_neighbour):
        nearest_neighbour_ids.append(distances[i][0])
    return nearest_neighbour_ids

In [13]:
get_k_neighbour(X_train[:5], X_test[1], 3)

[4, 2, 3]

In [14]:
def predict(X_train, X_test, y_train, num_neighbour = 3):
    y_predict = []
    for x_test in X_test:
        nearest_neighbour_ids = get_k_neighbour(X_train, x_test, num_neighbour)
        y_preds = y_train[nearest_neighbour_ids]
        y_preds = y_preds.mean()
        y_predict.append(y_preds)
    return y_predict

In [15]:
y_predict = predict(X_train[:30], X_test[:5], y_train[:30], num_neighbour = 5)
y_predict

[np.float64(4.4),
 np.float64(3.4),
 np.float64(2.8),
 np.float64(3.4),
 np.float64(3.4)]

### kNN в Sklearn

In [16]:
from sklearn.neighbors import KNeighborsRegressor

model = KNeighborsRegressor(n_neighbors = 5)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred

array([4.4, 4. , 3.2, 5.8, 4. , 4. , 4.6, 2.4, 4.6, 3.8, 2. , 5. , 3. ,
       5.2, 5.8, 5. , 2.2, 2.8, 4.8, 1.6, 3. ])

### Метрики

In [19]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score, mean_absolute_error

In [21]:
pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

MSE_train = mean_squared_error(y_train, pred_train)
RMSE_train = np.sqrt(MSE_train)
R2_train = r2_score(y_train, pred_train)
MAE_train = mean_absolute_error(y_train, pred_train)

MSE_test = mean_squared_error(y_test, pred_test)
RMSE_test = np.sqrt(MSE_test)
R2_test = r2_score(y_test, pred_test)
MAE_test = mean_absolute_error(y_test, pred_test)

print(f'MSE на обучении {MSE_train:.2f}')
print(f'MSE на тесте {MSE_test:.2f}', end='\n\n')

print(f'RMSE на обучении {RMSE_train:.2f}')
print(f'RMSE на тесте {RMSE_test:.2f}', end='\n\n')

print(f'R2 на обучении {R2_train:.2f}')
print(f'R2 на тесте {R2_test:.2f}', end='\n\n')

print(f'MAE на обучении {MAE_train:.2f}')
print(f'MAE на тесте {MAE_test:.2f}', end='\n\n')

MSE на обучении 2.56
MSE на тесте 1.82

RMSE на обучении 1.60
RMSE на тесте 1.35

R2 на обучении 0.33
R2 на тесте 0.31

MAE на обучении 1.32
MAE на тесте 1.15



### Decision Tree

In [23]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(random_state = 44)
model.fit(X_train, y_train) #обучени с методом .fit()

DecisionTreeRegressor(random_state=44)

In [24]:
pred_train = model.predict(X_train)
pred_test = model.predict(X_test)

MSE_train = mean_squared_error(y_train, pred_train)
RMSE_train = np.sqrt(MSE_train)
R2_train = r2_score(y_train, pred_train)
MAE_train = mean_absolute_error(y_train, pred_train)

MSE_test = mean_squared_error(y_test, pred_test)
RMSE_test = np.sqrt(MSE_test)
R2_test = r2_score(y_test, pred_test)
MAE_test = mean_absolute_error(y_test, pred_test)

print(f'MSE на обучении {MSE_train:.2f}')
print(f'MSE на тесте {MSE_test:.2f}', end='\n\n')

print(f'RMSE на обучении {RMSE_train:.2f}')
print(f'RMSE на тесте {RMSE_test:.2f}', end='\n\n')

print(f'R2 на обучении {R2_train:.2f}')
print(f'R2 на тесте {R2_test:.2f}', end='\n\n')

print(f'MAE на обучении {MAE_train:.2f}')
print(f'MAE на тесте {MAE_test:.2f}', end='\n\n')

MSE на обучении 0.00
MSE на тесте 4.05

RMSE на обучении 0.00
RMSE на тесте 2.01

R2 на обучении 1.00
R2 на тесте -0.55

MAE на обучении 0.00
MAE на тесте 1.57



Видно, что на обучающихся данных всё отлично, а в тестовых большие ошибки

### Переобучение:
#### Кросс - валидация (k-Fold)

In [28]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_validate

scores = cross_validate(DecisionTreeRegressor(random_state=44), X, y, cv = 5, # cv - это k частей
                       scoring={'r2': make_scorer(r2_score),
                                'mean_squared_error': make_scorer(mean_squared_error)},
                       return_train_score=True)

print('R2 train mean = ', scores['train_r2'].mean())
print('R2 test mean = ', scores['test_r2'].mean())

print('MSE train mean = ', scores['train_mean_squared_error'].mean())
print('MSE test mean = ', scores['test_mean_squared_error'].mean())

R2 train mean =  1.0
R2 test mean =  -0.7644861990897471
MSE train mean =  0.0
MSE test mean =  5.855714285714286


в данном случае увеличение датасета не помогло избавиться от переобучения.
Попробую другой метод борьбы с переобучением

#### Подбор гиперпараметров и GridSearchCV

In [30]:
# Какие гиперпараметры стоят
model.get_params()

{'ccp_alpha': 0.0,
 'criterion': 'squared_error',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': 44,
 'splitter': 'best'}

In [31]:
model = DecisionTreeRegressor(random_state = 44)
model.fit(X_train, y_train)

print(f'MSE train = {mean_squared_error(y_train, model.predict(X_train))}')
print(f'MSE test = {mean_squared_error(y_test, model.predict(X_test))}')

MSE train = 0.0
MSE test = 4.0476190476190474


In [33]:
# Если поменять max_depth

model = DecisionTreeRegressor(random_state = 1,
                             max_depth = 4,
                             min_samples_leaf = 1,
                             max_leaf_nodes = None)

model.fit(X_train, y_train)

print(f'MSE train = {mean_squared_error(y_train, model.predict(X_train))}')
print(f'MSE test = {mean_squared_error(y_test, model.predict(X_test))}')

MSE train = 1.514872408293461
MSE test = 2.2735170356053005


Видно, что MSE train не нулевое, и MSE test стало меньше, что говорит о том, что изменение гиперпараметров улучшает ситуацию

In [34]:
#а если поменять max_samples_leaf

model = DecisionTreeRegressor(random_state = 1,
                             max_depth = 4,
                             min_samples_leaf = 2,
                             max_leaf_nodes = None)

model.fit(X_train, y_train)

print(f'MSE train = {mean_squared_error(y_train, model.predict(X_train))}')
print(f'MSE test = {mean_squared_error(y_test, model.predict(X_test))}')

MSE train = 1.527372408293461
MSE test = 2.33304084512911


чуть ухудшилось качество, значит увеличение этого параметра не приводит ник чему хорошему

In [35]:
#а если поменять max_leaf_nodes

model = DecisionTreeRegressor(random_state = 1,
                             max_depth = 4,
                             min_samples_leaf = 1,
                             max_leaf_nodes = 3)

model.fit(X_train, y_train)

print(f'MSE train = {mean_squared_error(y_train, model.predict(X_train))}')
print(f'MSE test = {mean_squared_error(y_test, model.predict(X_test))}')

MSE train = 2.516119770303528
MSE test = 1.3145403322608513


Здесь интереснее, поскольку на train существенно ухудшилась метрика, а на test наоборот - значительно улучшилась.
Такой подбор "вручную" не очень удобен. Для этого в sklearn был реализован класс _GridSearchCV()_, который автоматически перебирает все возможные значения гиперпараметров по сетке

In [39]:
from sklearn.model_selection import GridSearchCV

model = DecisionTreeRegressor()

# описание сетки, по которой искать
param_grid = {
    #название параметра: значения, которые будем варьировать
    'max_depth': np.arange(1, 5),
    'min_samples_leaf': [1, 2, 3]
}

# создание объекта GridSearchCV
gridsearch = GridSearchCV(model, param_grid, refit = True, # заново обучаем с измененными гиперпараметрами
                          scoring=make_scorer(r2_score))

# запуск поиска по сетке всех гиперпараметров
gridsearch.fit(X_train, y_train)
print(gridsearch.best_params_)

best_model = gridsearch.best_estimator_

print(f'MSE train = {mean_squared_error(y_train, best_model.predict(X_train))}')
print(f'MSE test = {mean_squared_error(y_test, best_model.predict(X_test))}')

{'max_depth': np.int64(1), 'min_samples_leaf': 1}
MSE train = 2.8689168573607935
MSE test = 1.3723262544537203


Изначально было:<br>
MSE train = 0.0<br>
MSE test = 4.0476190476190474<br>
<br>
После _GridSearchCV()_ стало:<br>
MSE train = 2.8689168573607935<br>
MSE test = 1.3723262544537203

### Сохранение модели

In [40]:
import pickle

filename = 'finalized_model.sav'
pickle.dump(model, open(filename, 'wb'))

# load the model from disk
loaded_model = pickle.load(open(filename, 'rb'))